# ECG Arrhythmia Detection with Transformers - Colab Edition

This notebook runs the complete ECG Arrhythmia Transformer pipeline.

## Step 0: Install Dependencies

In [ ]:
!pip install -q torch numpy pandas scikit-learn scipy matplotlib seaborn wfdb pyyaml tqdm jupyter shap
print("[OK] Dependencies installed!")

## Step 1: Clone Repository

In [ ]:
import os
import subprocess
import sys

!git clone https://github.com/rajvardhansingh776/ecg-arrhythmia-transformer.git
os.chdir('ecg-arrhythmia-transformer')

print("[OK] Repository cloned!")

In [ ]:
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/tables', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("[OK] Directories created!")

## Step 2: Mount Google Drive (optional)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=True)
    print("[OK] Google Drive mounted!")
except:
    print("[WARN] Not in Colab, skipping Drive mount")

## Step 3: Preprocessing

In [ ]:
print("Running preprocessing...\n")
result = subprocess.run([sys.executable, 'preprocessing/preprocess_dataset.py'])
print(f"\nReturn code: {result.returncode}")

## Step 4: Self-Supervised Pretraining

In [ ]:
print("Running SSL pretraining...\n")
result = subprocess.run([sys.executable, 'training/pretrain_ssl.py'])
print(f"\nReturn code: {result.returncode}")

## Step 5: Model Training

In [ ]:
print("Running model training...\n")
result = subprocess.run([sys.executable, 'training/train.py'])
print(f"\nReturn code: {result.returncode}")

## Pipeline Summary

In [ ]:
from pathlib import Path

print("\n" + "="*50)
print("EXECUTION SUMMARY")
print("="*50 + "\n")

processed_dir = Path('data/processed')
if processed_dir.exists():
    data_files = list(processed_dir.glob('*.npy'))
    print(f"[OK] {len(data_files)} data files generated")

models_dir = Path('models')
if models_dir.exists():
    model_files = list(models_dir.glob('*.pt')) + list(models_dir.glob('*.pth'))
    if model_files:
        print(f"[OK] {len(model_files)} model(s) saved")

print("\nOutputs saved in:")
print("  - data/processed/  (preprocessed data)")
print("  - models/          (trained models)")
print("  - results/         (figures & tables)")
print("="*50)

## Backup Results to Google Drive

In [ ]:
import shutil

try:
    drive_path = '/content/gdrive/My Drive/ECG_Results'
    os.makedirs(drive_path, exist_ok=True)
    
    if os.path.exists('models'):
        dst = f'{drive_path}/models'
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree('models', dst)
        print("[OK] Models backed up to Drive")
    
    if os.path.exists('results'):
        dst = f'{drive_path}/results'
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree('results', dst)
        print("[OK] Results backed up to Drive")
    
    print("\nBackup location: /My Drive/ECG_Results")
except Exception as e:
    print(f"[WARN] Could not backup: {e}")